## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
# Packages
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from pyspark.ml.evaluation import RankingEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import ParamGridBuilder

In [2]:
# Build SparkSession
spark = (SparkSession.builder
         .appName("autoscrobbler")
         .config("spark.driver.memory", "12g")
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/23 18:16:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/23 18:16:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/23 18:16:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
# Load user_artist_data
user_artist_data_file = "/standard/ds7200-apt4c/large_datasets/user_artist_data.txt"

user_artist_full_df = spark.read.csv(user_artist_data_file,
                                sep=' ', 
                                header=False,
                                schema="userid INT, artistid INT, playcount INT")

In [4]:
# Sample user_artist_data
user_artist_sample_df = user_artist_full_df.sample(withReplacement=False,
                                            fraction=0.2,
                                            seed=314)

In [5]:
# Correct and collapse alternate artist aliases
# (I realize this is not required, but I wanted to do it as practice)
artist_alias_file = "artist_alias.txt"

artist_alias_df = spark.read.csv(artist_alias_file,
                                 sep='\t',
                                 header=False,
                                 schema="badid INT, goodid INT").dropna()

# Join dataframes
joined_df = user_artist_sample_df.join(
    artist_alias_df,
    on = user_artist_sample_df["artistid"] == artist_alias_df["badid"],
    how = "left")

# Replace incorrect artist aliases with goodid
aliased_df = joined_df.withColumn(
    "artistid",
    F.when(F.col("badid").isNotNull(), F.col("goodid"))
     .otherwise(F.col("artistid")))

# Sum user playcounts by artist
clean_df = (aliased_df
            .groupBy("userid", "artistid")
            .agg(F.sum("playcount").alias("playcount")))

# Count distinct artists per user and filter
count_df = (clean_df
            .groupBy("userid")
            .agg(F.countDistinct("artistid").alias("n_artists")))

filtered_df = (clean_df.join(
    count_df,
    on = "userid",
    how = "inner")
    .filter(F.col("n_artists") >= 10)
    .drop("n_artists"))

sorted_df = filtered_df.orderBy("userid", "artistid").cache()

In [6]:
# Split sample
train, validation, test = sorted_df.randomSplit([0.6, 0.2, 0.2], seed=314)

train.cache()
validation.cache()
test.cache()

print(train.count(), validation.count(), test.count())

sorted_df.unpersist()

2806574 934997 935876


DataFrame[userid: int, artistid: int, playcount: bigint]

In [7]:
# TASK 1 (1 PT)
# Show the first row from the training set
train.show(1)

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|    90|     121|        1|
+------+--------+---------+
only showing top 1 row


In [8]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as test_user1.
test.show(1)

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|    90|    1495|       60|
+------+--------+---------+
only showing top 1 row


In [9]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01
als = ALS(
    userCol = "userid",
    itemCol = "artistid",
    ratingCol = "playcount",
    implicitPrefs = True,
    rank = 8,
    maxIter = 5,
    regParam = 0.02,
    alpha = 0.01,
    seed = 314
)

model1 = als.fit(train)

In [10]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for test_user1
test_user1_df = test.select("userid").limit(1)
test_user1 = test_user1_df.first()["userid"]

recs1 = model1.recommendForUserSubset(test_user1_df, 10)
recs1.show(truncate=False)

[Stage 322:=======================================>              (73 + 9) / 100]

+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userid|recommendations                                                                                                                                                                                                       |
+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|90    |[{1000024, 0.018459074}, {1205, 0.018321734}, {930, 0.017852437}, {1001909, 0.017721875}, {1890, 0.016937176}, {1270639, 0.016479451}, {581, 0.014593793}, {976, 0.01451484}, {121, 0.014508895}, {1177, 0.013817517}]|
+------+------------------------------------------------------------------------------------------------

In [11]:
# TASK 4 (1 PT)
# Briefly explain Recall@K in the context of recommender systems

Recall@K is an expansion upon Recall, a metric equal to the number of true positives predicted divided by the total number of real positives.

Recall@K is a modified version of this, for the purpose of measuring performance of recommender systems which return k items. It is equal to the number of relevant items in the top k divided by the total number of relevant items. Relevancy can be defined in different ways, but when using implicit feedback, it could be any item with non-zero values or, when using a non-binary metric, a threshold may be set to filter incidental interactions.

Furthermore, it is only to be used for measuring performance on test or held-out data, as it will return optimistic results on training data. When used on test data, if it recommends items identified within the training data as relevant, those items may be dropped so as to avoid pessimistic results.

Lastly, if the count of a user's relevant items is greater than k, their Recall@k cannot reach 1, so scores for such users may be pessimistic as well.

In [12]:
# TASK 5 (1 PT)
# Calculate Recall@K for test_user1
# Relevant items are defined as the artists test_user1 played in the test set. 
# Recommendations are not filtered, so any training-set artists in the top 10 count as misses.
model1_pred_user1 = recs1.select(
    "userid",
    F.col("recommendations.artistid").cast("array<double>").alias("pred")
)

label_user1 = (test
            .filter(F.col("userid") == test_user1)
            .groupBy("userid")
            .agg(F.collect_list("artistid").cast("array<double>").alias("label")))

model1_eval_user1 = model1_pred_user1.join(label_user1, on="userid")

evaluator = RankingEvaluator(
    metricName = "recallAtK",
    k = 10,
    predictionCol = "pred",
    labelCol = "label"
)

model1_recall_user1 = evaluator.evaluate(model1_eval_user1)
print(model1_recall_user1)

0.0


In [13]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam, keep alpha=0.1) to maximize Recall@K
val_users = validation.select("userid").distinct().cache()
val_labels = (validation
              .groupBy("userid")
              .agg(F.collect_list("artistid").cast("array<double>").alias("label"))).cache()

valid_als = ALS(
    userCol="userid", 
    itemCol="artistid", 
    ratingCol="playcount",
    implicitPrefs=True,
    alpha=0.01,
    seed=314
)

param_grid = (ParamGridBuilder()
              .addGrid(valid_als.rank, list(range(8, 25, 8)))
              .addGrid(valid_als.maxIter, list(range(5, 11, 5)))
              .addGrid(valid_als.regParam, [0.02, 0.1])
              .build())

results = []

for params in param_grid:
    model = valid_als.fit(train, params)

    val_pred = (model.recommendForUserSubset(val_users, 10)
                .select("userid",
                        F.col("recommendations.artistid").cast("array<double>").alias("pred")))
    val_recall = evaluator.evaluate(val_pred.join(val_labels, on="userid"))

    combo = (params[valid_als.rank], params[valid_als.maxIter], params[valid_als.regParam])
    results.append((*combo, val_recall))
    print(*combo, val_recall)

8 5 0.02 0.021597959020202916


8 5 0.1 0.0237176469388458


8 10 0.02 0.024380333753700502


8 10 0.1 0.026048327995446544


16 5 0.02 0.01985535706629585


...

24 5 0.02 0.019003413775292657


24 5 0.1 0.020933226474340167


24 10 0.02 0.0211745466408911


[Stage 4905:>                                                       (0 + 8) / 9]

24 10 0.1 0.022426596591113145


In [14]:
best = max(results, key=lambda r: r[3])

In [15]:
# TASK 6 (2 PTS)
# Report the results including best values for (rank, maxIter, regParam) along with the Recall@K from the validation set
best_rank, best_maxIter, best_regParam, best_recall = best
print(f"Best: rank={best_rank}, maxIter={best_maxIter}, "
      f"regParam={best_regParam}, validation Recall@10={best_recall:.4f}")

Best: rank=8, maxIter=10, regParam=0.1, validation Recall@10=0.0260


In [16]:
# Retrain the best configuration on train + validation data.
tr_val_df = train.unionByName(validation)

final_als = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    rank=best_rank,
    maxIter=best_maxIter,
    regParam=best_regParam,
    alpha=0.01,
    seed=314
)

final_model = final_als.fit(tr_val_df)

In [17]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.
test_users = test.select("userid").distinct()
test_labels = (test
               .groupBy("userid")
               .agg(F.collect_list("artistid").cast("array<double>").alias("label")))

test_pred = (final_model.recommendForUserSubset(test_users, 10)
             .select("userid",
                     F.col("recommendations.artistid").cast("array<double>").alias("pred")))

test_recall = evaluator.evaluate(test_pred.join(test_labels, on="userid"))
print(f"Test Recall@10: {test_recall:.4f}")

[Stage 5423:>                                                       (0 + 8) / 9]

Test Recall@10: 0.0284


In [18]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?
final_recs1 = final_model.recommendForUserSubset(test_user1_df, 10)

final_pred_user1 = final_recs1.select(
    "userid",
    F.col("recommendations.artistid").cast("array<double>").alias("pred")
)

label_user1 = (test
            .filter(F.col("userid") == test_user1)
            .groupBy("userid")
            .agg(F.collect_list("artistid").cast("array<double>").alias("label")))

final_eval_user1 = final_pred_user1.join(label_user1, on="userid")

final_recall_user1 = evaluator.evaluate(final_eval_user1)
print(final_recall_user1)

0.0


In [19]:
final_pred_user1.show(truncate=False)

tr_val_df.filter(F.col("userid") == test_user1).orderBy(F.desc("playcount")).show(10)

test.filter(F.col("userid") == test_user1).orderBy(F.desc("playcount")).show(10)

known_user1 = {r["artistid"] for r in
               tr_val_df.filter(F.col("userid") == test_user1).select("artistid").collect()}
preds = final_pred_user1.first()["pred"]
print(len(known_user1), "known artists for test_user1")
print(len(label_user1.first()["label"]), "test artists for test_user1")
print(sum(int(a) in known_user1 for a in preds), "of 10 recommendations are already-known artists")

+------+-----------------------------------------------------------------------------------+
|userid|pred                                                                               |
+------+-----------------------------------------------------------------------------------+
|90    |[831.0, 121.0, 873.0, 1000009.0, 1014421.0, 1000113.0, 441.0, 478.0, 313.0, 5810.0]|
+------+-----------------------------------------------------------------------------------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|    90|    1208|       67|
|    90| 1014225|        9|
|    90|    1824|        2|
|    90|     121|        1|
|    90|     619|        1|
|    90| 1233511|        1|
|    90| 1233975|        1|
+------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|    90|    1495|       60|
|    90| 6720706|       30|
|    90|    2001|        1|
+------+--------+---------+

7 known artists for test_use

The recommendations for `test_user1` had the same Recall@k measure, meaning the tuning did not improve its recommendations. This may be due to the fact that `test_user1` happened to not have very many artists (10 total), and of those, they listened to half only a single time, so there was not much signal by which ALS might learn the user's preferences. Additionally, only three of `test_user1`'s artists were in the test set.

However, I would still say that hyperparameter tuning was of benefit to users on the whole, as there is a substantial difference in performance between both the baseline vs best iterations (baseline=0.0219, best=0.0258) and best vs worst tuning iterations (best=0.0258, worst=0.0195). The test recall (0.0277) was close to the best validation performance, demonstrating the tuned hyperparameters generalized well.